# mSigLIP Colab Training Experiments

Notebook này gom các bước cần để chạy training experiments trên Google Colab khi `VN3K/` và `m_siglip_checkpoints/` đã có sẵn trên Google Drive.

Workflow chính hiện tại:

1. Chạy **NACIR clean-safe trên VN3K sạch** để kiểm tra không làm giảm baseline.
2. Chạy **Circle vs NACIR trên synthetic FP/FN/FP+FN noise** để chứng minh robustness.
3. Sau khi robustness claim ổn, mới chạy **LoRA mạnh hơn / Part Align** để tìm hướng tăng R@1.

Notebook dùng TensorBoard mặc định, artifact/checkpoint lưu thẳng vào Drive để có thể resume khi Colab bị ngắt.


## 0. Runtime Check

Chạy cell này trước để xác nhận Colab đang dùng GPU runtime. Nếu `torch.cuda.is_available()` là `False`, vào `Runtime > Change runtime type > GPU` rồi chạy lại notebook.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)


def run_capture(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    proc = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc


def run_stream(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    proc_env = os.environ.copy()
    proc_env["PYTHONUNBUFFERED"] = "1"
    proc_env["HYDRA_FULL_ERROR"] = "1"
    proc_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    if env:
        proc_env.update(env)

    proc = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        env=proc_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)

    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {cmd}")
    return code


run("nvidia-smi", check=False)

try:
    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA:", torch.version.cuda)
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch import failed:", repr(exc))


## 1. Mount Drive + Path Config

Sửa các biến dưới đây nếu Drive của bạn đặt repo/data/model ở chỗ khác.

Hai layout phổ biến:

1. Chuẩn hóa:
   - `/content/drive/MyDrive/data/raw/VN3K/`
   - `/content/drive/MyDrive/artifacts/models/pretrained/m_siglip_checkpoints/model.safetensors`
2. Đặt trực tiếp dưới MyDrive:
   - `/content/drive/MyDrive/VN3K/`
   - `/content/drive/MyDrive/m_siglip_checkpoints/model.safetensors`

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print("Drive mount skipped or unavailable:", repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive')

# Edit these if needed.
PROJECT_DIR = DRIVE_ROOT / 'mSigLIP' / 'code'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data' / 'raw'
DRIVE_PRETRAINED_ROOT = DRIVE_ROOT / 'artifacts' / 'models' / 'pretrained'
DRIVE_ARTIFACTS_ROOT = DRIVE_ROOT / 'msiglip_colab' / 'artifacts'

# Fallback layout: uncomment if VN3K and m_siglip_checkpoints are directly under MyDrive.
# DRIVE_DATA_ROOT = DRIVE_ROOT
# DRIVE_PRETRAINED_ROOT = DRIVE_ROOT

os.environ['MSIGLIP_DATA_ROOT'] = str(DRIVE_DATA_ROOT)
os.environ['MSIGLIP_PRETRAINED_ROOT'] = str(DRIVE_PRETRAINED_ROOT)
os.environ['MSIGLIP_ARTIFACTS_ROOT'] = str(DRIVE_ARTIFACTS_ROOT)
DRIVE_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR           =', PROJECT_DIR)
print('MSIGLIP_DATA_ROOT     =', os.environ['MSIGLIP_DATA_ROOT'])
print('MSIGLIP_PRETRAINED_ROOT =', os.environ['MSIGLIP_PRETRAINED_ROOT'])
print('MSIGLIP_ARTIFACTS_ROOT  =', os.environ['MSIGLIP_ARTIFACTS_ROOT'])

## 2. Repo Setup

Cell này chuyển vào repo và cài package ở chế độ editable. Notebook dùng `pip`, không dùng `uv`, vì Colab đã có Python runtime riêng.

Nếu Colab báo thiếu package, giữ `INSTALL_MINIMAL_DEPS=True`. Nếu dependency đã đầy đủ và muốn nhanh hơn, đổi thành `False`.

In [ ]:
import os
import sys
from pathlib import Path

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print('cwd =', Path.cwd())

# PROJECT_DIR must be the full mSigLIP repo, not just a folder containing this notebook, VN3K, and checkpoints.
core_repo_files = [
    PROJECT_DIR / 'pyproject.toml',
    PROJECT_DIR / 'trainer.py',
    PROJECT_DIR / 'src' / 'msiglip' / 'train.py',
    PROJECT_DIR / 'configs' / 'cir_msiglip.yaml',
    PROJECT_DIR / 'configs' / 'loss' / 'cir_msiglip.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'default.yaml',
]
missing_core = [str(path) for path in core_repo_files if not path.exists()]
if missing_core:
    print('PROJECT_DIR is not a complete/current repo. Missing core files:')
    for path in missing_core:
        print(' -', path)
    raise FileNotFoundError(
        'Upload/sync the full repo to Google Drive or set PROJECT_DIR to the real repo folder. '
        'VN3K and m_siglip_checkpoints alone are not enough to run training.'
    )

INSTALL_MINIMAL_DEPS = True

# Make local src importable immediately in this kernel and subprocesses. This is
# enough for this repo because trainer.py also inserts PROJECT_DIR/src itself.
src_dir = str(PROJECT_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
os.environ['PYTHONPATH'] = src_dir + os.pathsep + os.environ.get('PYTHONPATH', '')

if INSTALL_MINIMAL_DEPS:
    # Keep torch from the Colab runtime. Install only project-side dependencies commonly missing in Colab.
    run(
        f"{sys.executable} -m pip install -q "
        "hydra-core omegaconf lightning loguru prettytable peft safetensors "
        "transformers sentencepiece ftfy tensorboard wandb scikit-learn scipy seaborn matplotlib nltk"
    )
    # Colab may preinstall an old torchao. PEFT 0.19 refuses torchao<0.16 even for normal LoRA.
    # We do not use torchao here, so uninstalling it is safer than upgrading the Colab Torch stack.
    run(f"{sys.executable} -m pip uninstall -y -q torchao", check=False)


# Editable install is convenient but not required for this notebook. On some
# Colab/Python/setuptools combinations it can fail while direct PYTHONPATH works.
editable = run(f"{sys.executable} -m pip install -e . --no-deps", check=False)
if editable.returncode != 0:
    print("WARNING: editable install failed; continuing with PYTHONPATH fallback:", src_dir)
    print("If you want the exact pip error, rerun: python -m pip install -e . --no-deps -v")

# The text augmentation module initializes NLTK stopwords at import time.
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
except Exception as exc:
    print('NLTK data setup warning:', repr(exc))

print('Setup complete')

## 3. Asset Verification

Fail sớm nếu data/model path sai. Cần thấy:

- `$MSIGLIP_DATA_ROOT/VN3K`
- `$MSIGLIP_PRETRAINED_ROOT/m_siglip_checkpoints/model.safetensors`

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(os.environ['MSIGLIP_DATA_ROOT'])
PRETRAINED_ROOT = Path(os.environ['MSIGLIP_PRETRAINED_ROOT'])
ARTIFACTS_ROOT = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT'])

vn3k_dir = DATA_ROOT / 'VN3K'
model_path = PRETRAINED_ROOT / 'm_siglip_checkpoints' / 'model.safetensors'

print('VN3K dir:', vn3k_dir)
print('Model:', model_path)
print('Artifacts:', ARTIFACTS_ROOT)

assert vn3k_dir.exists(), f"Missing VN3K dir: {vn3k_dir}"
assert model_path.exists(), f"Missing mSigLIP checkpoint: {model_path}"

annotation_files = sorted(vn3k_dir.glob('data_captions*.json'))
image_samples = sorted(vn3k_dir.rglob('*.jpg'))[:5] + sorted(vn3k_dir.rglob('*.png'))[:5]

print('Annotation files:')
for path in annotation_files[:10]:
    print(' -', path)
print('Image samples:')
for path in image_samples[:10]:
    print(' -', path)

assert annotation_files, f"No data_captions*.json found under {vn3k_dir}"
assert image_samples, f"No image samples found under {vn3k_dir}"

## 4. Preflight Cho NACIR

Chạy unit tests nhanh cho NACIR clean-safe và synthetic noise injection. Không dùng `fast_dev_run` ở đây vì retrieval validation cần đủ image/text positives; `fast_dev_run` cắt validation quá ngắn và có thể tạo lỗi metric giả.


In [ ]:
required_files = [
    PROJECT_DIR / 'tests' / 'test_nacir_clean_safe.py',
    PROJECT_DIR / 'tests' / 'test_noise_injection.py',
    PROJECT_DIR / 'configs' / 'loss' / 'cir_msiglip.yaml',
    PROJECT_DIR / 'configs' / 'lora' / 'default.yaml',
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    print('Missing required repo files:')
    for path in missing:
        print(' -', path)
    raise FileNotFoundError('Colab repo is missing recent NACIR/noise-injection files. Sync/upload the latest training package.')

run_capture(
    f'{sys.executable} -c \'import peft, hydra, omegaconf; '
    'print("peft", peft.__version__); print("hydra ok"); print("omegaconf ok")\''
)
run_capture(f"{sys.executable} -m unittest discover -s tests -p 'test_nacir_clean_safe.py' -v")
run_capture(f"{sys.executable} -m unittest discover -s tests -p 'test_noise_injection.py' -v")


## 5. NACIR Clean VN3K

Chạy trước để kiểm tra NACIR clean-safe không làm giảm hiệu năng trên VN3K sạch. Cấu hình này giữ training batch giống baseline (`batch_size=24`, `accumulate_grad_batches=3`) và chỉ tăng `test_batch_size` để A100 evaluate nhanh hơn.

Acceptance tạm thời: `test_t2i_R1 >= 51.8`.


In [ ]:
# PEFT trên Colab có thể vấp torchao cũ. LoRA thường không cần torchao.
run_capture(f"{sys.executable} -m pip uninstall -y torchao", check=False)

clean_nacir_cmd = " ".join([
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    "trainer.max_epochs=60",
    "dataset.batch_size=24",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    "trainer.accumulate_grad_batches=3",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.NACIR=true",
    "loss.PART_ALIGN=false",
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
    "logger.experiment_name=nacir_clean_vn3k_a100_fair",
    "+lora=default",
])

print(clean_nacir_cmd)


In [ ]:
run_stream(clean_nacir_cmd)


## 6. NACIR Robustness Trên Synthetic Noise

Chạy sau khi clean NACIR pass. Mục tiêu là so sánh cùng một mapping noise giữa Circle và NACIR.

- FP noise: shuffle caption positive bằng `dataset.noisy_rate`.
- FN noise: tách một phần sample cùng PID sang fake PID bằng `dataset.fn_noisy_rate`.
- FP+FN: bật cả hai.

Noise index `.npy` sẽ tự tạo nếu chưa tồn tại và được reuse để so sánh công bằng.


In [ ]:
from pathlib import Path
import os
import sys

NOISE_DIR = Path(os.environ["MSIGLIP_ARTIFACTS_ROOT"]) / "training" / "noiseindex"
NOISE_DIR.mkdir(parents=True, exist_ok=True)

ROBUST_COMMON = [
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    "trainer.max_epochs=60",
    "dataset.batch_size=24",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    "trainer.accumulate_grad_batches=3",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.PART_ALIGN=false",
    "+lora=default",
]


def robust_cmd(name, nacir, extra):
    return " ".join(
        ROBUST_COMMON
        + [
            f"logger.experiment_name={name}",
            f"loss.NACIR={'true' if nacir else 'false'}",
        ]
        + extra
    )

fp04 = [
    "dataset.noisy_rate=0.4",
    f"dataset.noisy_file={NOISE_DIR / 'VN3K_VI_FP_0.4.npy'}",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
]

fn04 = [
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.4",
    f"dataset.fn_noisy_file={NOISE_DIR / 'VN3K_VI_FN_0.4.npy'}",
]

fpfn04 = [
    "dataset.noisy_rate=0.4",
    f"dataset.noisy_file={NOISE_DIR / 'VN3K_VI_FP_0.4.npy'}",
    "dataset.fn_noisy_rate=0.4",
    f"dataset.fn_noisy_file={NOISE_DIR / 'VN3K_VI_FN_0.4.npy'}",
]

robustness_commands = {
    "circle_fp04": robust_cmd("circle_fp04_a100", False, fp04),
    "nacir_fp04": robust_cmd("nacir_fp04_a100", True, fp04),
    "circle_fn04": robust_cmd("circle_fn04_a100", False, fn04),
    "nacir_fn04": robust_cmd("nacir_fn04_a100", True, fn04),
    "circle_fpfn04": robust_cmd("circle_fpfn04_a100", False, fpfn04),
    "nacir_fpfn04": robust_cmd("nacir_fpfn04_a100", True, fpfn04),
}

ROBUSTNESS_RUN_ORDER = [
    "circle_fp04",
    "nacir_fp04",
    "circle_fn04",
    "nacir_fn04",
    "circle_fpfn04",
    "nacir_fpfn04",
]

for key in ROBUSTNESS_RUN_ORDER:
    print(f"\n# {key}\n{robustness_commands[key]}")


In [ ]:
# Chạy từng experiment một. Đổi RUN_NAME theo ROBUSTNESS_RUN_ORDER ở cell trên.
RUN_NAME = "circle_fp04"
run_stream(robustness_commands[RUN_NAME])


## 7. Phương Pháp Mới Để Tăng R@1

Chỉ chạy section này sau khi NACIR clean + robustness đã có kết quả. Các lệnh dưới đây nhằm tăng năng lực biểu diễn, tách khỏi claim robustness.

Khuyến nghị thứ tự: `circle_attn_ffn_r32_a100` trước, rồi `circle_attn_ffn_r64_a100` hoặc `circle_part_align_r32_a100` nếu còn runtime.


In [ ]:
STRONG_COMMON = [
    f"{sys.executable} -u trainer.py -cn cir_msiglip",
    "trainer.max_epochs=60",
    "dataset.batch_size=48",
    "dataset.test_batch_size=128",
    "dataset.num_workers=4",
    "trainer.accumulate_grad_batches=2",
    "++trainer.precision=16-mixed",
    "logger.logger_type=tensorboard",
    "optimizer=cir_test",
    "optimizer.param_groups.default.lr=1e-4",
    "loss.NACIR=false",
    "dataset.noisy_rate=0.0",
    "dataset.noisy_file=null",
    "dataset.fn_noisy_rate=0.0",
    "dataset.fn_noisy_file=null",
]


def strong_cmd(name, *extra):
    return " ".join(STRONG_COMMON + [f"logger.experiment_name={name}"] + list(extra))

strong_commands = {
    "circle_attn_ffn_r32_a100": strong_cmd(
        "circle_attn_ffn_r32_a100", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32"
    ),
    "circle_attn_ffn_r64_a100": strong_cmd(
        "circle_attn_ffn_r64_a100", "loss.PART_ALIGN=false", "+lora=attn_ffn_r64"
    ),
    "circle_attn_ffn_r32_rslora_a100": strong_cmd(
        "circle_attn_ffn_r32_rslora_a100", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32_rslora"
    ),
    "circle_attn_ffn_r32_pissa_a100": strong_cmd(
        "circle_attn_ffn_r32_pissa_a100", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32_pissa"
    ),
    "circle_attn_ffn_r32_dora_a100": strong_cmd(
        "circle_attn_ffn_r32_dora_a100", "loss.PART_ALIGN=false", "+lora=attn_ffn_r32_dora"
    ),
    "circle_part_align_r32_a100": strong_cmd(
        "circle_part_align_r32_a100", "loss.PART_ALIGN=true", "+lora=attn_ffn_r32"
    ),
    "nacir_attn_ffn_r32_a100_optional": " ".join(
        STRONG_COMMON
        + [
            "logger.experiment_name=nacir_attn_ffn_r32_a100_optional",
            "loss.NACIR=true",
            "loss.PART_ALIGN=false",
            "+lora=attn_ffn_r32",
        ]
    ),
}

for key, cmd in strong_commands.items():
    print(f"\n# {key}\n{cmd}")


In [ ]:
# Chạy từng experiment một sau khi robustness đã xong.
RUN_NAME = "circle_attn_ffn_r32_a100"
run_stream(strong_commands[RUN_NAME])


## 8. Resume + Monitor

Dùng section này nếu Colab disconnect hoặc hết phiên. Checkpoint `last.ckpt` được lưu dưới Drive artifacts. Chọn đúng base command đang chạy rồi append `+ckpt_path=...`.


In [ ]:
from pathlib import Path
import os

runs_dir = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs'
last_ckpts = sorted(
    runs_dir.glob('**/last.ckpt'),
    key=lambda p: p.stat().st_mtime if p.exists() else 0,
)

if last_ckpts:
    latest_ckpt = last_ckpts[-1]
    print('Latest last.ckpt:', latest_ckpt)
    print('\nRecent checkpoints:')
    for path in last_ckpts[-10:]:
        print(path)
else:
    latest_ckpt = None
    print('No last.ckpt found under', runs_dir)


def resume_cmd(base_cmd, ckpt_path=None):
    ckpt = ckpt_path or latest_ckpt
    if ckpt is None:
        raise FileNotFoundError(f'No last.ckpt found under {runs_dir}')
    return f"{base_cmd} +ckpt_path={ckpt}"


In [ ]:
# Ví dụ resume clean NACIR. Đổi base command nếu đang resume robustness/new-method run.
if latest_ckpt is not None:
    resume_clean_nacir_cmd = resume_cmd(clean_nacir_cmd)
    print(resume_clean_nacir_cmd)
    # run_stream(resume_clean_nacir_cmd)


In [ ]:
# TensorBoard. If the magic has trouble with env vars, paste the printed path manually.
print('TensorBoard logdir:', Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs')


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$MSIGLIP_ARTIFACTS_ROOT/training/runs"


## 9. Result Collection

Cell này liệt kê log/config/checkpoint quan trọng để tải về hoặc ghi vào journal sau full run.


In [ ]:
from pathlib import Path
import os

root = Path(os.environ['MSIGLIP_ARTIFACTS_ROOT']) / 'training' / 'runs'
patterns = ['**/train.log', '**/.hydra/config.yaml', '**/checkpoints/*.ckpt', '**/events.out.tfevents.*']

for pattern in patterns:
    matches = sorted(root.glob(pattern), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    print(f"\n# {pattern} ({len(matches)} files)")
    for path in matches[-20:]:
        print(path)

print('\nSau khi có full-run result, ghi metric vào docs/journal/[train]-YYYY-MM-DD.md theo policy repo.')
